# Notebook 11 — Analyse des scores LLM-as-judge

**Entrée** : un ou plusieurs fichiers produits par le notebook 10 (`results/llm_judge_*.json`), au format `{ "summary": ..., "details": [...] }` avec les scores **cohérence**, **utilité**, **fidélité** (1–5).

**Sorties** : tableaux pandas (moyennes, écarts-types, % valides), figures dans `results/plots/` (`fig_judge_*.png`). **Aucun appel API** — lecture locale uniquement.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
BASE_PATH = "/content/drive/MyDrive/llm-integration-study/"

In [ ]:
!pip install -q pandas matplotlib

In [ ]:
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RESULTS_PATH   = os.path.join(BASE_PATH, "results")
PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")
PLOTS_PATH     = os.path.join(RESULTS_PATH, "plots")
os.makedirs(PLOTS_PATH, exist_ok=True)

SCORE_KEYS = ("coherence", "utilite", "fidelite")

# Liste explicite (prioritaire) ou None pour tout prendre results/llm_judge_*.json
JUDGE_FILES = None  # ex: [os.path.join(RESULTS_PATH, "llm_judge_rag.json")]

def discover_judge_files():
    if JUDGE_FILES:
        return [p for p in JUDGE_FILES if os.path.isfile(p)]
    pattern = os.path.join(RESULTS_PATH, "llm_judge_*.json")
    return sorted(glob.glob(pattern))


def load_judge_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return {
            "summary": {"predictions_label": os.path.basename(path), "judge_model": "?", "n": len(data)},
            "details": data,
        }
    return data


def details_to_df(payload: dict, source_path: str) -> pd.DataFrame:
    summ = payload.get("summary") or {}
    label_default = summ.get("predictions_label") or os.path.basename(source_path)
    judge_model = summ.get("judge_model") or summ.get("model_judge") or "?"
    rows = payload.get("details") or []
    rec = []
    for r in rows:
        method = r.get("predictions_label") or label_default
        row = {
            "pair_id": str(r.get("pair_id", "")),
            "method": method,
            "judge_model": r.get("model_judge") or judge_model,
            "source_file": os.path.basename(source_path),
        }
        for k in SCORE_KEYS:
            v = r.get(k)
            row[k] = int(v) if isinstance(v, (int, float)) and not isinstance(v, bool) and 1 <= int(v) <= 5 else np.nan
        row["parse_ok"] = all(row[k] == row[k] for k in SCORE_KEYS)
        rec.append(row)
    return pd.DataFrame(rec)


paths = discover_judge_files()
if not paths:
    raise FileNotFoundError(
        "Aucun fichier llm_judge_*.json dans results/. Lance le notebook 10 ou renseigne JUDGE_FILES."
    )

dfs = []
for p in paths:
    dfs.append(details_to_df(load_judge_json(p), p))
df = pd.concat(dfs, ignore_index=True)
print(f"{len(paths)} fichier(s) juge | {len(df)} lignes | méthodes : {df['method'].nunique()}")
display(df.head())

In [ ]:
# Résumé numérique par méthode (réponses juge uniquement)
agg_rows = []
for method, g in df.groupby("method"):
    row = {"method": method, "n_rows": len(g), "n_valid": int(g["parse_ok"].sum())}
    for k in SCORE_KEYS:
        s = g[k].dropna()
        row[f"mean_{k}"] = round(float(s.mean()), 3) if len(s) else np.nan
        row[f"std_{k}"] = round(float(s.std(ddof=0)), 3) if len(s) > 1 else (0.0 if len(s) == 1 else np.nan)
        row[f"median_{k}"] = float(s.median()) if len(s) else np.nan
    # Score composite (moyenne des 3 critères sur les lignes où les 3 sont présents)
    trip = g[list(SCORE_KEYS)].dropna()
    row["mean_composite"] = round(float(trip.mean(axis=1).mean()), 3) if len(trip) else np.nan
    agg_rows.append(row)

summary_df = pd.DataFrame(agg_rows).sort_values("mean_composite", ascending=False, na_position="last")
print("=== Synthèse LLM-as-judge par méthode ===")
display(summary_df)

out_csv = os.path.join(RESULTS_PATH, "llm_judge_summary_table.csv")
summary_df.to_csv(out_csv, index=False)
print(f"CSV : {out_csv}")

In [ ]:
# Figure 1 — Moyennes des 3 critères par méthode (barres groupées)
methods = summary_df["method"].tolist()
x = np.arange(len(methods))
w = 0.22
labels_fr = {"coherence": "Cohérence", "utilite": "Utilité", "fidelite": "Fidélité"}
colors = {"coherence": "#4C72B0", "utilite": "#55A868", "fidelite": "#C44E52"}

fig, ax = plt.subplots(figsize=(max(8, len(methods) * 1.2), 5))
for i, k in enumerate(SCORE_KEYS):
    offs = (i - 1) * w
    vals = [summary_df.loc[summary_df["method"] == m, f"mean_{k}"].values[0] for m in methods]
    ax.bar(x + offs, vals, w, label=labels_fr[k], color=colors[k], alpha=0.88)

ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=22, ha="right")
ax.set_ylabel("Score moyen (1–5)")
ax.set_ylim(0, 5.5)
ax.axhline(3.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
ax.legend(title="Critère")
ax.set_title("LLM-as-judge — moyennes par méthode")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
p1 = os.path.join(PLOTS_PATH, "fig_judge_means_by_method.png")
fig.savefig(p1, dpi=150, bbox_inches="tight")
plt.show()
print(p1)

In [ ]:
# Figure 2 — Distribution des scores (normalisée) par critère et par méthode
labels_fr = {"coherence": "Cohérence", "utilite": "Utilité", "fidelite": "Fidélité"}
fig2, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
for ax, k in zip(axes, SCORE_KEYS):
    for method in df["method"].unique():
        sub = df.loc[df["method"] == method, k].dropna().astype(int)
        if sub.empty:
            continue
        counts = sub.value_counts().reindex(range(1, 6), fill_value=0)
        pct = counts / counts.sum() * 100
        ax.plot(range(1, 6), pct.values, marker="o", label=method)
    ax.set_xticks(range(1, 6))
    ax.set_xlabel("Score")
    ax.set_ylabel("% des réponses")
    ax.set_title(labels_fr[k])
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=8, loc="upper left")
fig2.suptitle("LLM-as-judge — distribution des scores 1–5 (% par méthode)")
fig2.tight_layout()
p2 = os.path.join(PLOTS_PATH, "fig_judge_score_distributions.png")
fig2.savefig(p2, dpi=150, bbox_inches="tight")
plt.show()
print(p2)

In [ ]:
# Figure 3 — Violon / composite : une ligne = une réponse évaluée
plot_df = df.dropna(subset=list(SCORE_KEYS))
plot_df = plot_df.copy()
plot_df["composite"] = plot_df[list(SCORE_KEYS)].mean(axis=1)

methods_ord = summary_df["method"].tolist()
methods_plot = [m for m in methods_ord if len(plot_df.loc[plot_df["method"] == m]) > 0]
data_v = [plot_df.loc[plot_df["method"] == m, "composite"].values for m in methods_plot]

fig3, ax3 = plt.subplots(figsize=(max(7, len(methods_plot)), 5))
ax3.violinplot(data_v, positions=range(1, len(methods_plot) + 1), showmeans=True, showmedians=False)
ax3.set_xticks(range(1, len(methods_plot) + 1))
ax3.set_xticklabels(methods_plot, rotation=20, ha="right")
ax3.set_ylabel("Score composite (moyenne des 3 critères)")
ax3.set_ylim(0.5, 5.5)
ax3.set_title("LLM-as-judge — dispersion du score composite par méthode")
ax3.grid(axis="y", alpha=0.3)
fig3.tight_layout()
p3 = os.path.join(PLOTS_PATH, "fig_judge_composite_violin.png")
fig3.savefig(p3, dpi=150, bbox_inches="tight")
plt.show()
print(p3)

In [ ]:
# Optionnel — croisement avec test.json (composite par strate temporelle)
test_path = os.path.join(PROCESSED_PATH, "test.json")
if os.path.isfile(test_path):
    with open(test_path, "r", encoding="utf-8") as f:
        test_rows = json.load(f)
    meta = pd.DataFrame(test_rows)
    meta["pair_id"] = meta["pair_id"].astype(str)
    merge_cols = ["pair_id"]
    for c in ("recency_category", "question_type", "dataset_type"):
        if c in meta.columns:
            merge_cols.append(c)
    merged = df.merge(meta[merge_cols], on="pair_id", how="left")

    def canon_recency(raw):
        if raw is None or (isinstance(raw, float) and np.isnan(raw)):
            return "inconnu"
        k = str(raw).strip().lower()
        alias = {"recent": "récent", "récent": "récent", "intermediaire": "intermédiaire", "intermédiaire": "intermédiaire", "fundamental": "fondamental", "fondamental": "fondamental"}
        return alias.get(k, k or "inconnu")

    if "recency_category" in merged.columns:
        merged["recency"] = merged["recency_category"].apply(canon_recency)
    else:
        merged["recency"] = "inconnu"

    strata = ["récent", "intermédiaire", "fondamental", "inconnu"]
    strata = [s for s in strata if (merged["recency"] == s).any()]

    fig4, ax4 = plt.subplots(figsize=(max(9, len(summary_df) * 1.1), 5))
    meths = summary_df["method"].tolist()
    xh = np.arange(len(meths))
    bw = 0.22
    pal = ["#e74c3c", "#f39c12", "#2ecc71", "#95a5a6"]
    for i, st in enumerate(strata):
        vals = []
        for m in meths:
            sub = merged.loc[(merged["method"] == m) & (merged["recency"] == st), list(SCORE_KEYS)].dropna(how="any")
            vals.append(float(sub.mean(axis=1).mean()) if len(sub) else np.nan)
        ax4.bar(xh + (i - len(strata) / 2 + 0.5) * bw, vals, bw, label=st, color=pal[i % len(pal)], alpha=0.85)
    ax4.set_xticks(xh)
    ax4.set_xticklabels(meths, rotation=20, ha="right")
    ax4.set_ylabel("Composite moyen (1–5)")
    ax4.set_title("LLM-as-judge — composite par strate temporelle")
    ax4.legend(title="Strate")
    ax4.set_ylim(0, 5.5)
    ax4.grid(axis="y", alpha=0.3)
    fig4.tight_layout()
    p4 = os.path.join(PLOTS_PATH, "fig_judge_composite_by_recency.png")
    fig4.savefig(p4, dpi=150, bbox_inches="tight")
    plt.show()
    print(p4)
else:
    print("test.json absent — figure stratifiée ignorée.")